# Freezer

**What's in this notebook?** This notebook demonstrates the module **`freezer`** — a framework for freezing out heavy moduli from the EFT. The `ConifoldFreezer` integrates out the conifold modulus $z_{\text{cf}}$ by solving its leading-order EOM analytically.

**In this notebook, you will learn:**
- How to set up a `ConifoldFreezer` from an existing `FluxEFT` model
- How the conifold modulus is solved analytically and frozen out
- How to compute the reduced F-terms $D_\alpha W_{\text{bulk}}$ in the light-field EFT
- How to write a custom `Freezer` subclass for other moduli

**Prerequisites:** [Tutorial 13: coniLCS](../04_geometry_and_limits/13_coniLCS.ipynb) for conifold geometry setup.

We use the $h^{1,1}=99$, $h^{1,2}=3$ example from [2009.03312](https://arxiv.org/pdf/2009.03312) throughout.

(*Created:* March 2026)

## Imports

In [ ]:
# General imports
import warnings
import numpy as np
from scipy.optimize import root

# JAX imports
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# JAXVacua
import jaxvacua as jvc
from jaxvacua.flux_utils import flux_to_pfv, pfv_to_flux, pfv_to_moduli
from jaxvacua.freezer import Freezer, ConifoldFreezer

warnings.filterwarnings('ignore')

## Model setup

We use the $h^{1,1}=99$, $h^{1,2}=3$ coniLCS model from [2009.03312](https://arxiv.org/pdf/2009.03312), with the conifold curve $[-1,1,0]$.

In [ ]:
from cytools import Polytope

poly = Polytope([[-1,3,-2,-1],[1,-1,0,0],[-1,0,0,1],[-1,0,0,0],[-1,0,1,1],[-1,0,2,0],[-1,0,1,0]])
cy = poly.triangulate().get_cy()

basis_matrix = np.array([[0, 1, 1], [1, 1, 0], [0, 0, 1]])
conifold_curve = np.array([-1, 1, 0])

model = jvc.FluxEFT(
    h12=cy.h11(), Q=cy.h11()+cy.h12()+2, 
    use_cytools=True, mirror_cy=cy, ncf=2, use_gvs=True,
    maximum_degree=6, basis_change=basis_matrix, 
    limit="coniLCS", prange=10, conifold_basis=True
)
model

## `freezer` — freezing out heavy moduli

### The idea

When a modulus is parametrically heavy, its leading-order EOM determines it as a function of the remaining light fields. We can substitute this solution back into the superpotential and work with an EFT that has fewer degrees of freedom.

The `Freezer` base class defines the general interface. Subclasses implement:
- `heavy_indices` — which moduli to freeze out
- `solve_heavy` — how to solve for them given light fields
- `_real_light_to_full` — how to convert real coordinates

The base class then provides `reconstruct_full_moduli`, `superpotential`, `DW_light`, `DW_x_light`, and `dDW_x_light` for free.

### `ConifoldFreezer`: freezing out $z_{\text{cf}}$

Near the conifold locus, $z_{\text{cf}}$ acquires a parametrically large mass. Its leading-order EOM gives

$$z_{\text{cf}} = -\frac{1}{2\pi i}\exp\!\left(-\frac{2\pi i\,\widetilde{W}_1}{n_{\text{cf}}(M_1 - \tau H_1)}\right)$$

**TODO: add more details from notes here as to what define $\widetilde{W}_1$ using the notation from the code!!! Also add corrections!**

The `ConifoldFreezer` implements this formula. Let's set it up:

In [ ]:
freezer = ConifoldFreezer(model, conifold_index=0)

print(f"Heavy indices: {freezer.heavy_indices}")
print(f"Light indices: {freezer.light_indices}")
print(f"Freezing out {freezer.n_heavy} modulus, keeping {freezer.n_light} light moduli + tau")
print(f"Conifold degree (sourced from lcs_tree.conifold.ncf): {freezer.ncf}")

### Solving for $z_{\text{cf}}$ from the bulk moduli

Given the bulk moduli and $\tau$, the freezer solves for $z_{\text{cf}}$. The available `mode` values mirror the underlying `compute_zcf` dispatcher in `conifold_utils`:

- `mode="manual"` (default) — closed-form $\widetilde{W}_1$ assembly from `kappa` / `a_matrix` / `b_vector` plus the worldsheet-instanton $\mathrm{Li}$ sums.
- `mode="autodiff"` — same $\widetilde{W}_1$ via the css-side `F_coniLCS_exp` + `dF_coniLCS_exp` (must agree with `"manual"` to numerical precision).
- `mode="pfv"` — PFV / linear-racetrack approximation; uses only the integer fluxes and becomes exact at the racetrack-stationary point.

In [ ]:
# Use the first solution from Table 1 of arXiv:2009.03312 as the running example
M = np.array([4, -8, 8])
K = np.array([-8, 3, -6])
tau0 = 1j / 0.38

# PFV initial guess for the bulk moduli
z0 = pfv_to_moduli(model, M, K, tau0)
zbulk = z0[1:]  # light (bulk) moduli
flux = pfv_to_flux(model, M, K)

# Solve for z_cf via the closed-form (manual) route through W_log_coeff.
zcf_full = freezer.solve_heavy(zbulk, tau0, flux, mode="manual")

# Solve for z_cf via the PFV / linear-racetrack approximation.
zcf_pfv = freezer.solve_heavy(zbulk, tau0, flux, mode="pfv")

# Cross-check directly against the model's `compute_zcf_x` dispatcher.
# `compute_zcf_x` takes the bulk-only real vector (length 2*h12, no z_cf
# direction); slicing `x[2:]` strips Re(z_cf) / Im(z_cf) from the full real x.
x_full = model._convert_complex_to_real(z0, jnp.conj(z0), tau0, jnp.conj(tau0))
x_bulk = x_full[2:]

zcf_old_pfv = model.compute_zcf_x(x_bulk, flux, mode="pfv")
zcf_old_full = model.compute_zcf_x(x_bulk, flux, mode="manual")

print("ConifoldFreezer (full):    ", np.abs(zcf_full[0]))
print("model.compute_zcf_x (full):", np.abs(zcf_old_full))
print()
print("ConifoldFreezer (PFV):     ", np.abs(zcf_pfv[0]))
print("model.compute_zcf_x (PFV): ", np.abs(zcf_old_pfv))

### Reconstructing full moduli and computing the superpotential

The freezer can reconstruct the full moduli vector (inserting the solved $z_{\text{cf}}$) and evaluate EFT quantities on the reduced field space:

In [ ]:
# Reconstruct full moduli from light ones
z_full = freezer.reconstruct_full_moduli(zbulk, tau0, flux)
print(f"Full moduli (freezer):  {z_full}")
print(f"Full moduli (PFV):      {z0}")
print()

# Superpotential evaluated with z_cf on-shell
W_reduced = freezer.superpotential(zbulk, tau0, flux)
W_full = model.superpotential(z0, tau0, flux)
print(f"|W| (reduced EFT): {np.abs(W_reduced):.6e}")
print(f"|W| (full PFV):    {np.abs(W_full):.6e}")

### Using the freezer for optimisation

The key use case: solve for the true vacuum using only the bulk moduli as free variables. We first find the full vacuum, then show the freezer reproduces $z_{\text{cf}}$ at the solution.

In [ ]:
# Find the true vacuum using the full model (all moduli free)
x0 = model._convert_complex_to_real(z0, jnp.conj(z0), tau0, jnp.conj(tau0))
res = root(model.DW_x, x0, args=(flux,), jac=model.dDW_x, tol=1e-10, method="hybr")

x_sol = res.x
z_sol, _, tau_sol, _ = model._convert_real_to_complex(x_sol)

print(f"Full minimisation converged: {res.success}")
print(f"sum|DW|: {np.sum(np.abs(model.DW_x(x_sol, flux))):.2e}")
print(f"z_cf (numerical):  {z_sol[0]}")
print(f"|z_cf| (numerical): {np.abs(z_sol[0]):.6e}")

Now we use the freezer to predict $z_{\text{cf}}$ from the bulk moduli at the solution and compare with the numerics:

In [ ]:
# Use the freezer to predict z_cf from the solved bulk moduli
zbulk_sol = z_sol[1:]
zcf_frozen = freezer.solve_heavy(zbulk_sol, tau_sol, flux)

print(f"|z_cf| (numerical):         {np.abs(z_sol[0]):.10e}")
print(f"|z_cf| (freezer, full):     {np.abs(zcf_frozen[0]):.10e}")
print(f"|z_cf| (freezer, PFV):      {np.abs(freezer.solve_heavy(zbulk_sol, tau_sol, flux, mode='pfv')[0]):.10e}")
print()
print(f"Relative error (full):  {np.abs(np.abs(z_sol[0]) - np.abs(zcf_frozen[0])) / np.abs(z_sol[0]):.2e}")

### Reduced DW: F-terms for the light moduli only

The freezer provides `DW_light` — the covariant derivatives $D_i W$ restricted to the light moduli and $D_\tau W$, with $z_{\text{cf}}$ frozen out. At the true vacuum, these should be close to zero:

In [ ]:
# DW for the full model at the solution (all 4 components: 3 moduli + tau)
DW_full = model.DW(z_sol, jnp.conj(z_sol), tau_sol, jnp.conj(tau_sol), flux)
print("Full DW (all moduli + tau):")
print(f"  |DW| = {np.abs(DW_full)}")
print()

# DW from the freezer (only 2 bulk moduli + tau = 3 components)
DW_light = freezer.DW_light(
    zbulk_sol, jnp.conj(zbulk_sol), tau_sol, jnp.conj(tau_sol), flux
)
print("Reduced DW (light moduli + tau only, z_cf frozen):")
print(f"  |DW| = {np.abs(DW_light)}")
print(f"  Dimensions: {len(DW_full)} (full) → {len(DW_light)} (reduced)")

### Real-coordinate interface: `DW_x_light` and `dDW_x_light`

For use with `scipy.optimize.root` or similar solvers, the freezer also provides real-coordinate versions. These operate on a reduced real vector of length $2 \times n_{\text{light}} + 2$ (bulk moduli + tau) instead of the full $2 \times h^{1,2} + 2$:

In [ ]:
# Full real DW at the solution
DW_x_full = model.DW_x(x_sol, flux)
print(f"Full DW_x ({len(DW_x_full)} components):")
print(f"  |DW_x| = {np.abs(DW_x_full)}")
print()

# Reduced real DW — only bulk moduli + tau (6 components instead of 8)
x_bulk = x_sol[2:]  # drop the first two entries (Re(z_cf), Im(z_cf))
DW_x_light = freezer.DW_x_light(x_bulk, flux)
print(f"Reduced DW_x_light ({len(DW_x_light)} components):")
print(f"  |DW_x| = {np.abs(DW_x_light)}")

# Cross-check directly against the (now-deprecated) `model.DWbulk_x` —
# kept as a compatibility check while `private/promotion/vacuum_promotion.py`
# is being migrated to the freezer interface.  Emits DeprecationWarning.
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    DW_x_bulk_old = model.DWbulk_x(x_bulk, flux, mode="manual")
print(f"\nmodel.DWbulk_x (deprecated; {len(DW_x_bulk_old)} components):")
print(f"  |DW_x| = {np.abs(DW_x_bulk_old)}")
print(f"\nMatch: {np.allclose(DW_x_light, DW_x_bulk_old)}")

### Solving the reduced F-term conditions with `scipy`

We can directly use `DW_x_light` and `dDW_x_light` as objective and Jacobian for `scipy.optimize.root`, solving only for the 6 bulk + tau real variables instead of 8:

In [ ]:
# Initial guess: PFV values for bulk moduli + tau (drop z_cf)
x0_bulk = model._convert_complex_to_real(z0, jnp.conj(z0), tau0, jnp.conj(tau0))[2:]

# Solve the reduced system
res_reduced = root(
    lambda x: freezer.DW_x_light(x, flux),
    x0_bulk,
    jac=lambda x: freezer.dDW_x_light(x, flux),
    tol=1e-10,
    method="hybr"
)

print(f"Reduced minimisation converged: {res_reduced.success}")
print(f"Variables: {len(x0_bulk)} (reduced) vs {len(x0)} (full)")
print(f"sum|DW_x|: {np.sum(np.abs(res_reduced.fun)):.2e}")
print(f"nfev: {res_reduced.nfev}")

Compare the solution from the reduced system with the full minimisation:

In [ ]:
# Recover full moduli from the reduced solution
x_red = res_reduced.x
x_full_reconstructed = np.array(freezer._real_light_to_full(x_red, flux))
z_red, _, tau_red, _ = model._convert_real_to_complex(x_full_reconstructed)

print("Bulk moduli comparison (full vs reduced):")
for i in range(1, model.h12):
    print(f"  z[{i}]: {z_sol[i]:.10f}  vs  {z_red[i]:.10f}")

print(f"\ntau: {tau_sol:.10f}  vs  {tau_red:.10f}")
print(f"\n|z_cf| (full):    {np.abs(z_sol[0]):.10e}")
print(f"|z_cf| (reduced): {np.abs(z_red[0]):.10e}")

# Check W0
W_red = model.superpotential(z_red, tau_red, flux, normalise=True)
W_full = model.superpotential(z_sol, tau_sol, flux, normalise=True)
print(f"\n|W0| (full):    {np.abs(W_full):.10e}")
print(f"|W0| (reduced): {np.abs(W_red):.10e}")

### Validation across multiple flux choices

Let's verify the freezer against all five examples from Table 1 in [2009.03312](https://arxiv.org/pdf/2009.03312):

In [ ]:
Mlist = np.array([[4,-8,8], [4,-8,10], [8,-12,6], [-8,4,12], [-14,6,27]])
Klist = np.array([[-8,3,-6], [-6,3,-4], [-5,1,-2], [5,1,-4], [4,1,-2]])
gslist = np.array([0.38, 0.15, 0.125, 0.35, 0.0643])

print(f"{'#':>2} | {'converged':>9} | {'|z_cf| num':>14} | {'|z_cf| frozen':>14} | {'rel err':>10} | {'|W0|':>12}")
print("-" * 85)

for i in range(len(Mlist)):
    Mi, Ki, gsi = Mlist[i], Klist[i], gslist[i]
    
    # Build flux and initial guess via flux_utils
    fluxi = pfv_to_flux(Mi, Ki, tree)
    tau_i = 1j / gsi
    z_i = pfv_to_moduli(Mi, Ki, tau_i, tree)
    
    # Full minimisation
    x_i = model._convert_complex_to_real(z_i, jnp.conj(z_i), tau_i, jnp.conj(tau_i))
    res_i = root(model.DW_x, x_i, args=(fluxi,), jac=model.dDW_x, tol=1e-10, method="hybr")
    
    if not res_i.success:
        # Fallback: Newton + retry
        z_tmp, _, tau_tmp, _ = model._convert_real_to_complex(res_i.x)
        z_tmp, tau_tmp, _ = model.newton_method_flux_vacua(
            z_tmp, tau_tmp, fluxi, step_size_Newton=1, tol=1e-12, max_iters=10, mode="SUSY", solver_mode="real"
        )
        x_i = model._convert_complex_to_real(z_tmp, jnp.conj(z_tmp), tau_tmp, jnp.conj(tau_tmp))
        res_i = root(model.DW_x, x_i, args=(fluxi,), jac=model.dDW_x, tol=1e-10, method="hybr")
    
    z_i_sol, _, tau_i_sol, _ = model._convert_real_to_complex(res_i.x)
    
    # Freezer prediction
    zcf_frozen_i = freezer.solve_heavy(z_i_sol[1:], tau_i_sol, fluxi)
    
    zcf_num = np.abs(z_i_sol[0])
    zcf_frz = np.abs(zcf_frozen_i[0])
    rel_err = np.abs(zcf_num - zcf_frz) / zcf_num
    W0 = np.abs(model.superpotential(z_i_sol, tau_i_sol, fluxi, normalise=True))
    
    print(f"{i:>2} | {str(res_i.success):>9} | {zcf_num:>14.6e} | {zcf_frz:>14.6e} | {rel_err:>10.2e} | {W0:>12.6e}")

### Writing a custom `Freezer` subclass

The `Freezer` base class is designed to be extended. To freeze out a different set of heavy moduli, subclass `Freezer` and implement `heavy_indices`, `solve_heavy`, and `_real_light_to_full`. Here is a minimal skeleton:

```python
from jaxvacua.freezer import Freezer

class MyCustomFreezer(Freezer):
    
    @property
    def heavy_indices(self):
        return (0, 1)  # freeze out the first two moduli
    
    def solve_heavy(self, z_light, tau, fluxes, **kwargs):
        # Your analytic (or numerical) solution for the heavy moduli
        # as functions of z_light, tau, and fluxes
        z_heavy = ...  
        return z_heavy
    
    def _real_light_to_full(self, x_light, fluxes, **kwargs):
        # Convert real light coordinates to full real array
        # by solving for heavy moduli and inserting them
        ...
        return x_full
```

The base class then automatically provides `superpotential`, `DW_light`, `DW_x_light`, and `dDW_x_light` for the reduced theory.